# Test-Set Evaluation

Standalone evaluation of optimized prompts against held-out test sets.

**Prerequisites:** TermNorm backend running | Datasets created via optimization notebook | `.env` with API keys

Shares `dataset_runs/` with the optimization notebook — all eval data is content-hash deduped.

In [ ]:
%load_ext autoreload
%autoreload 2

import json, os

from _campaign_lib import *

svc = await init_services()

In [2]:
#@title Backend status
backend_status = await show_backend_status(svc["backend_client"])

2026-03-03 18:36:40 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/status "HTTP/1.1 200 OK"


BACKEND STATUS
  Session Active                 False
  Active Sessions                0
  Terms Loaded                   0
  Match Database Identifiers     108
  Match Database Aliases         489
  Experiments Count              4
  Mappings Count                 1126
  Pipeline Version               v1.1
  Llm Provider                   groq
  Llm Model                      meta-llama/llama-4-maverick-17b-128e-instruct


## Data Flow

**Session terms** = the candidate pool loaded into TermNorm for fuzzy matching + LLM ranking.

- Must include **all** `ground_truth` identifiers from train + test datasets
- Train has `query→ground_truth` mappings (used for optimization); test has identifiers only (realistic candidate pool)
- This notebook initializes the TermNorm session with the full candidate pool before running any evaluation

In [ ]:
#@title Dataset summary
ds_summary = show_dataset_summary(svc["store"], svc["backend_id"])
all_session_terms = ds_summary["all_session_terms"]

In [ ]:
#@title Load test datasets
test_processes = load_stored_dataset(svc["store"], svc["backend_id"], "test_processes")
test_material = load_stored_dataset(svc["store"], svc["backend_id"], "test_material")

if not test_processes and not test_material:
    print("\nNo test datasets found. Run load_or_create_datasets() in the optimization notebook first.")

In [ ]:
#@title Load optimized prompt
# Option 1: Load from campaign winner file
from api.services.stores.base import read_json_optional
from api.models.prompt_state import PromptState
from pathlib import Path

# Find most recent winner
opt_dir = svc["store"].base_dir / svc["backend_id"] / "sync" / "optimization"
winner_files = sorted(opt_dir.glob("campaign_winner_*.json")) if opt_dir.exists() else []

if winner_files:
    winner_data = read_json_optional(winner_files[-1])
    prompt = PromptState(**winner_data["winner"])
    print(f"Loaded winner: {winner_files[-1].name}")
    print(f"  Accuracy: {winner_data['accuracy']:.1%}")
    print(f"  Prompt ID: {prompt.id[:12]}")
else:
    # Option 2: Fall back to baseline from experiment
    prompt = load_baseline_prompt(svc["exp_data"]) if svc.get("exp_data") else None
    if prompt:
        print(f"No winner found — using baseline prompt: {prompt.id[:12]}")
    else:
        print("No prompt available. Define one manually below.")

if prompt:
    rendered = prompt.render()
    print(f"\nRendered prompt ({len(rendered)} chars):")
    print(rendered[:300])
    if len(rendered) > 300:
        print("...")

In [ ]:
#@title Evaluation config
eval_config = {
    "model": "meta-llama/llama-4-maverick-17b-128e-instruct",
    "temperature": 0,
}

# Pipeline config (from backend)
pipeline_config = load_pipeline_config(svc["exp_data"]) if svc.get("exp_data") else {"steps": []}
pipeline_params = build_pipeline_params(pipeline_config)

print(f"Pipeline params: {pipeline_params}")
print(f"Session terms: {len(all_session_terms)} identifiers")

In [ ]:
#@title Initialize TermNorm session
svc["backend_client"].init_session(all_session_terms)
print(f"Session initialized with {len(all_session_terms)} terms")

In [ ]:
#@title Evaluate: test_processes
from api.services.prompt_eval import evaluate_prompt_cached

if test_processes and prompt:
    proc_results, proc_scores, proc_cached = await evaluate_prompt_cached(
        prompt, test_processes, svc["backend_client"],
        pipeline_params=pipeline_params,
        store=svc["store"], backend_id=svc["backend_id"],
        label="test_processes",
        model=eval_config["model"],
        temperature=eval_config["temperature"],
    )
    cached_str = " [cached]" if proc_cached else ""
    print(f"test_processes: {proc_scores['accuracy']:.1%} "
          f"({proc_scores['hits']}/{proc_scores['total']}){cached_str}")
else:
    print("Skipped — no test_processes data or prompt.")
    proc_results, proc_scores = [], {}

In [ ]:
#@title Evaluate: test_material
if test_material and prompt:
    mat_results, mat_scores, mat_cached = await evaluate_prompt_cached(
        prompt, test_material, svc["backend_client"],
        pipeline_params=pipeline_params,
        store=svc["store"], backend_id=svc["backend_id"],
        label="test_material",
        model=eval_config["model"],
        temperature=eval_config["temperature"],
    )
    cached_str = " [cached]" if mat_cached else ""
    print(f"test_material: {mat_scores['accuracy']:.1%} "
          f"({mat_scores['hits']}/{mat_scores['total']}){cached_str}")
else:
    print("Skipped — no test_material data or prompt.")
    mat_results, mat_scores = [], {}

In [ ]:
#@title Results summary
import pandas as pd

rows = []
for name, scores, results in [
    ("test_processes", proc_scores, proc_results),
    ("test_material", mat_scores, mat_results),
]:
    if scores:
        rows.append({
            "test_set": name,
            "accuracy": f"{scores['accuracy']:.1%}",
            "hits": scores["hits"],
            "total": scores["total"],
            "errors": scores.get("errors", 0),
        })

if rows:
    print("EVALUATION RESULTS")
    print("=" * 60)
    display(pd.DataFrame(rows))

    # Per-query breakdown
    all_results = proc_results + mat_results
    if all_results:
        misses = [r for r in all_results if not r.get("hit") and r.get("status") == "success"]
        print(f"\nMISSES ({len(misses)}/{len(all_results)})")
        print("-" * 60)
        for r in misses[:15]:
            print(f"  {r['query'][:50]:<50s}")
            print(f"    pred: {r['predicted'][:40]}")
            print(f"    gt:   {r['ground_truth'][:40]}")
else:
    print("No evaluation results to display.")